In [0]:
import requests

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

from pyspark.sql.functions import (
    col,
    explode,
    array,
    split,
    current_date
)

In [0]:
api_url = "https://reqres.in/api/users"

all_data = []
page = 1

while True:
    response = requests.get(
        api_url,
        params={"page": page},
        timeout=30
    )

    response.raise_for_status()

    result = response.json()
    users = result.get("data", [])

    if not users:
        break

    all_data.extend(users)
    page += 1

print("Total users fetched:", len(all_data))

In [0]:
print(all_data[:2])

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

user_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("email", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("avatar", StringType(), True)
])

In [0]:
df = spark.createDataFrame(
    all_data,
    schema=user_schema
)

display(df)

In [0]:
df_flat = df.withColumn(
    "site_address",
    split(col("email"), "@")[1]
)

display(df_flat)

In [0]:
df_flat = df_flat.withColumn(
    "load_date",
    current_date()
)

display(df_flat)

In [0]:
output_path = "/Volumes/main/default/assignment_data/site_info/person_info"

(
    df_flat.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .save(output_path)
)

In [0]:
result_df = (
    spark.read
    .format("delta")
    .load(output_path)
)

display(result_df)